# SoulX-FlashHead: Talking Head Generator

Go to **Runtime** > **Change runtime type** > **T4 GPU** before running.

In [ ]:
# @title 1. Setup Environment
!git clone https://github.com/Soul-AILab/SoulX-FlashHead.git
%cd SoulX-FlashHead

!apt-get install -y ffmpeg -q
!pip install -q xformers
!pip install -q "gradio>=5.0.0" diffusers transformers accelerate tqdm imageio easydict ftfy imageio-ffmpeg scikit-image loguru pyloudnorm decord librosa flask huggingface_hub
!pip install -q "xfuser>=0.4.3" yunchang distvae sentencepiece beautifulsoup4 einops

# Fix for face detection (replaces broken mediapipe with OpenCV)
import os
os.makedirs("flash_head/utils", exist_ok=True)
with open("flash_head/utils/cpu_face_handler.py", "w") as f:
    f.write('''import cv2\nimport numpy as np\nimport os\nfrom typing import Tuple, List\n\nclass CPUFaceHandler:\n    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):\n        proto = "/content/SoulX-FlashHead/deploy.prototxt"\n        caffe = "/content/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"\n        self.use_dnn = False\n        import urllib.request\n        if not os.path.exists(proto):\n            try: urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt", proto)\n            except: pass\n        if not os.path.exists(caffe):\n            try: urllib.request.urlretrieve("https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel", caffe)\n            except: pass\n        if os.path.exists(proto) and os.path.exists(caffe):\n            self.net = cv2.dnn.readNetFromCaffe(proto, caffe)\n            self.use_dnn = True\n        else:\n            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n            self.cascade = cv2.CascadeClassifier(cascade_path)\n\n    def detect(self, image: np.ndarray) -> Tuple[List, List]:\n        bboxs, scores = [], []\n        img_h, img_w = image.shape[:2]\n        if self.use_dnn:\n            blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))\n            self.net.setInput(blob)\n            detections = self.net.forward()\n            for i in range(detections.shape[2]):\n                confidence = float(detections[0, 0, i, 2])\n                if confidence > 0.5:\n                    bboxs.append([float(detections[0, 0, i, 3]), float(detections[0, 0, i, 4]), float(detections[0, 0, i, 5]), float(detections[0, 0, i, 6])])\n                    scores.append(confidence)\n        else:\n            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)\n            faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))\n            for (x, y, w, h) in faces:\n                bboxs.append([x/img_w, y/img_h, (x+w)/img_w, (y+h)/img_h]); scores.append(1.0)\n        return bboxs, scores\n\n    def __call__(self, image: np.ndarray) -> Tuple[List, List]:\n        return self.detect(image)\n''')
print("✅ Setup complete")

In [ ]:
# @title 2. Download Models
from huggingface_hub import snapshot_download
import os

os.makedirs("/content/models", exist_ok=True)

print("📥 Downloading SoulX-FlashHead (1.3B)... may take a few mins")
snapshot_download(
    repo_id="Soul-AILab/SoulX-FlashHead-1_3B",
    local_dir="/content/models/SoulX-FlashHead-1_3B",
    local_dir_use_symlinks=False
)

print("📥 Downloading wav2vec2-base-960h...")
snapshot_download(
    repo_id="facebook/wav2vec2-base-960h",
    local_dir="/content/models/wav2vec2-base-960h",
    local_dir_use_symlinks=False
)

print("✅ Models downloaded successfully!")

In [ ]:
# @title 3. Start Launch App
import os
with open('gradio_app.py', 'r') as f:
    code = f.read()

code = code.replace('value="models/SoulX-FlashHead-1_3B"', 'value="/content/models/SoulX-FlashHead-1_3B"')
code = code.replace('value="models/wav2vec2-base-960h"', 'value="/content/models/wav2vec2-base-960h"')
code = code.replace('app.launch()', 'app.launch(share=True, debug=True)')

with open('gradio_app_simple.py', 'w') as f:
    f.write(code)

!python gradio_app_simple.py